# Active Rods

---

- Many real swimmers are elongated and swim along their long axis: bacteria such as *E. coli* and *B. subtilis*, gliding myxobacteria, and synthetic catalytic rods
- Combining self-propulsion with an elongated shape gives new collective behaviour: when two rods collide they turn towards each other and align, which can produce clusters, swarms, lanes and 'active turbulence' that neither active spheres nor passive rods show
- We build each rod as a rigid chain of beads (as in `rod.ipynb`) and make the head bead an ABP: it pushes the rod along its axis with force $F_0$, and that direction diffuses rotationally
- In 2D, each rod $i$ with centre $\mathbf{r}_i$ and angle $\theta_i$ (so $\hat{\mathbf{e}}_i = (\cos\theta_i, \sin\theta_i)$) follows:

$$\dot{\mathbf{r}}_i = \frac{1}{\gamma}\left(\mathbf{F}_i^{\text{Ext}} + F_0\hat{\mathbf{e}}_i\right) + \sqrt{2D_t}\,\boldsymbol{\xi}_i \qquad \dot{\theta}_i = \frac{\tau_i}{\gamma_r} + \sqrt{2D_r}\,\eta_i$$

- $\mathbf{F}_i^{\text{Ext}}$ and $\tau_i$ are the force and torque from collisions with other rods; the torque is what makes rods align
- The main parameters to explore are the activity ($F_0$ and $D_r$, combined in the Péclet number), the area fraction $\phi$ and the rod aspect ratio

In [ ]:
import hoomd # simulation engine
import numpy as np # arrays and random numbers
import gsd.hoomd # reading and writing HOOMD trajectory files
import os # deleting old files

### Activity parameters
- Each rod swims at speed $v_0 = F_0/\gamma$ ($\gamma = n$ here) and keeps its direction for a time $\sim 1/D_r$, so it travels a persistence length $\ell_p = v_0/D_r$ before turning
- The Péclet number $\mathrm{Pe} = v_0/(\sigma D_r)$ compares swimming with diffusion; with $\phi$ and the aspect ratio it is the main control parameter (papers define it with slightly different factors)

In [ ]:
N = 100 # number of rods
n = 5 # number of beads in each rod
b = 0.5 # spacing between beads in a rod
phi = 0.5 # area fraction: fraction of the box covered by rods
kT = 1.0 # temperature (thermal energy)
F0 = 20.0 # self-propulsion force on each head bead (swim speed v0 = F0 / n)
D_r = 0.4 # rotational diffusion constant of the swimming direction (ABP)

In [ ]:
# shape of one rod in its own frame: n beads along the x axis, centred on the origin
bead_x = (np.arange(n) - (n - 1) / 2) * b # x position of each bead
bead_positions = [(x, 0.0, 0.0) for x in bead_x]
bead_types = ['A'] * (n - 1) + ['H'] # the last bead (at the +x end) is the head H, so the rod points towards its head
rod_length = (n - 1) * b # distance from the first bead centre to the last
I_z = np.sum(bead_x**2) # moment of inertia about z (each bead has mass 1)

a_rod = rod_length * 1 + np.pi / 4 # area of one rod: a strip 1 wide plus two half-disc end caps
L = np.sqrt(N * a_rod / phi) # size of our 2D box, chosen to give area fraction phi
A = L**2 # Area of our 2D box

In [ ]:
# place N rods on a lattice: rows of rods lying along the x axis
nx = max(1, int(round(np.sqrt(N / (rod_length + 1))))) # rods per row, so the gaps along and across the rods scale with the rod shape
ny = int(np.ceil(N / nx)) # number of rows
dx = L / nx # spacing between rod centres along a row
dy = L / ny # spacing between rows
if dx < rod_length + 1 or dy < 1:
    raise ValueError("phi is too high for this lattice, the rods would overlap")

i, j = np.meshgrid(np.arange(nx), np.arange(ny)) # lattice site indices
positions = np.zeros((N, 3)) # x, y, z of each rod centre (z stays 0 in 2D)
positions[:, 0] = (i.ravel()[:N] + 0.5) * dx - L / 2 # x in [-L/2, L/2)
positions[:, 1] = (j.ravel()[:N] + 0.5) * dy - L / 2 # y in [-L/2, L/2)

# point each rod along +x or -x at random, so they don't all face the same way
rng = np.random.default_rng() # random number generator
theta = rng.choice([0, np.pi], size=N) # angle of each rod
orientations = np.zeros((N, 4)) # quaternion for a rotation by theta about z
orientations[:, 0] = np.cos(theta / 2)
orientations[:, 3] = np.sin(theta / 2)

In [ ]:
# build the initial frame: only the rod centres, HOOMD adds the beads later
frame = gsd.hoomd.Frame()
frame.particles.N = N
frame.particles.types = ['R', 'A', 'H'] # R = rod centre, A = body bead, H = head bead (A and H must be listed even though there are none yet)
frame.particles.typeid = np.zeros(N, dtype=int) # every particle is a rod centre (type R)
frame.particles.position = positions
frame.particles.orientation = orientations
frame.particles.mass = n * np.ones(N) # mass of the whole rod
frame.particles.moment_inertia = np.tile([0, 0, I_z], (N, 1)) # only rotation about z in 2D
frame.configuration.box = [L, L, 0, 0, 0, 0] # Lx, Ly, Lz, xy, xz, yz (Lz = 0 makes it 2D)

In [ ]:
# delete any old initial config file
init_filename = "active_rods_init.gsd"
if os.path.exists(init_filename):
    os.remove(init_filename)

# save the initial config ("x" creates a new file)
with gsd.hoomd.open(name=init_filename, mode="x") as f:
    f.append(frame)

In [ ]:
GPU = hoomd.device.GPU() # use CPU() instead if you don't have a GPU
simulation = hoomd.Simulation(device=GPU, seed=1) # seed sets the random numbers
simulation.create_state_from_gsd(filename=init_filename) # load the rod centres

In [ ]:
# define the rod shape and add the beads to every rod
rigid = hoomd.md.constrain.Rigid()
rigid.body['R'] = {
    "constituent_types": bead_types, # body beads are type A, the head bead is type H
    "positions": bead_positions, # bead positions relative to the rod centre
    "orientations": [(1.0, 0.0, 0.0, 0.0)] * n, # beads are not rotated relative to the rod
}
rigid.create_bodies(simulation.state) # adds n beads to every rod, only call this once

In [ ]:
integrator = hoomd.md.Integrator(dt=1e-4, integrate_rotational_dof=True) # rotational_dof lets the rods rotate
integrator.rigid = rigid # keep the beads fixed in each rod

In [ ]:
# Brownian dynamics for the rod centres only, the beads just follow their rod
Brownian = hoomd.md.methods.Brownian(filter=hoomd.filter.Rigid(("center", "free")), kT=kT)
Brownian.gamma['R'] = n # translational drag: each bead adds drag 1
Brownian.gamma_r['R'] = (1.0, 1.0, kT / D_r) # rotational drag chosen so the rods rotate diffusively with D_r = kT / gamma_r (x and y must not be 0, but are unused in 2D)
integrator.methods.append(Brownian)

In [ ]:
cell = hoomd.md.nlist.Cell(buffer=0.4, exclusions=('bond', 'body')) # 'body': beads in the same rod don't interact

In [ ]:
# WCA (purely repulsive LJ) between beads, set r_cut = 2.5 instead for attractive LJ
# the head H interacts the same as the body beads A, change the H pairs to give the head different interactions
lj = hoomd.md.pair.LJ(nlist=cell, mode='shift')
lj.params[(['A', 'H'], ['A', 'H'])] = dict(epsilon=1.0, sigma=1.0) # A-A, A-H and H-H
lj.r_cut[(['A', 'H'], ['A', 'H'])] = 2**(1/6)
lj.params[('R', ['R', 'A', 'H'])] = dict(epsilon=0.0, sigma=1.0) # rod centres don't interact
lj.r_cut[('R', ['R', 'A', 'H'])] = 0.0
integrator.forces.append(lj)

### Why activity is interesting
- Active particles constantly use energy, so they are out of equilibrium: purely repulsive ABPs can clump into dense clusters (motility-induced phase separation, MIPS), which can't happen in equilibrium
- For rods, collisions also turn them towards each other, giving clusters, swarms, lanes or 'active turbulence' depending on Pe, $\phi$ and the aspect ratio

In [ ]:
# self-propulsion: a constant force F0 on each head bead H, pointing along the rod towards the head
# Rigid passes this force on to the whole rod, so each rod swims head first
# this makes each head an ABP: it swims with force F0 along its direction, and that direction diffuses with D_r
# because Brownian dynamics randomly rotates each rod (gamma_r = kT / D_r, set in the Brownian cell)
# so we don't need HOOMD's separate rotational diffusion updater (it would double count the rotation)
active = hoomd.md.force.Active(filter=hoomd.filter.Type(['H'])) # only the head beads are active
active.active_force['H'] = (F0, 0.0, 0.0) # force in the head bead's own frame: +x is along the rod, towards the head
active.active_torque['H'] = (0.0, 0.0, 0.0) # no active torque
integrator.forces.append(active)

In [ ]:
simulation.operations.integrator = integrator # attach the integrator to the simulation

In [ ]:
# delete any old trajectory file
trajectory_filename = "active_rods_trajectory.gsd"
if os.path.exists(trajectory_filename):
    os.remove(trajectory_filename)

# save the trajectory to file
gsd_writer = hoomd.write.GSD(filename=trajectory_filename,
                            trigger=hoomd.trigger.Periodic(period=1000), # save a frame every 1000 steps
                            mode="wb", # overwrite any existing file
                            filter=hoomd.filter.All(), # save rod centres and beads
                            dynamic=['property', 'particles/image']) # save positions, orientations and box crossings every frame
simulation.operations.writers.append(gsd_writer) # attach the writer to the simulation

In [ ]:
simulation.run(100000) # run for 100,000 steps
gsd_writer.flush() # write any frames still held in memory to the file